In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN

from sklearn.metrics import classification_report, confusion_matrix

# 1. 데이터 로드
df = pd.read_csv('creditcard.csv')

In [4]:
# 2. 데이터 확인 0 정상  1 비정상
print(df.head())
print(df['Class'].value_counts())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [5]:
# 3. 특징 / 라벨 분리
X = df.drop(['Class'], axis=1)
y = df['Class']

In [6]:
# 4. 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Isolation Forest
model_1 = IsolationForest(
    n_estimators=100,
    contamination=0.0017,  # 이상치 비율 반영 0.17% 
    random_state=42
)

model_1_pred = model_1.fit_predict(X_scaled)

# 결과 변환: (정상:1 → 0, 이상:-1 → 1)
#model_1_pred = np.where(model_1_pred == -1, 1, 0)
model_1_pred = [1 if x == -1 else 0 for x in model_1_pred]

In [7]:
# 6. DBSCAN
dbscan_sample = df.sample(frac=0.1, random_state=42)
X_sample = dbscan_sample.drop(['Class'], axis=1)
y_sample = dbscan_sample['Class']

scaler = StandardScaler()
X_sample_scaled = scaler.fit_transform(X_sample)

model_2 = DBSCAN(eps=5, min_samples=5)
model_2_pred = model_2.fit_predict(X_sample_scaled)

# DBSCAN: 정상=클러스터 번호(0 이상), 이상치=-1
#model_2_pred = np.where(model_2_pred == -1, 1, 0)
model_2_pred = [1 if x == -1 else 0 for x in model_2_pred]


In [9]:
print("\n Isolation Forest")
print(confusion_matrix(y, model_1_pred))
print(classification_report(y, model_1_pred))

print("\n DBSCAN")
print(confusion_matrix(y_sample, model_2_pred))
print(classification_report(y_sample, model_2_pred))


 Isolation Forest
[[283955    360]
 [   367    125]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    284315
           1       0.26      0.25      0.26       492

    accuracy                           1.00    284807
   macro avg       0.63      0.63      0.63    284807
weighted avg       1.00      1.00      1.00    284807


 DBSCAN
[[27702   733]
 [    8    38]]
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     28435
           1       0.05      0.83      0.09        46

    accuracy                           0.97     28481
   macro avg       0.52      0.90      0.54     28481
weighted avg       1.00      0.97      0.99     28481



Isolation Forest  
이상치 492개중 125개만 탐지 약 75프로의 값을 제대로 탐지하지 못함  
이상치라고 판단한 것중 약 74% 오탐  


DBSCAN  
이상치 46개중 38개 탐지 약 83프로 제대로 탐지  
이상치라고 한 것중 95프로 오탐
이상치는 잘 찾지만 대신 오탐이 매우 많음(정싱을 이상치로 판단)



isolation Forest는 정상적인 데이터에 높은 정확도와 안정적인 성능을 나타내지만 이상치 탐지율은 한계를 보임    
DBSCAN은 이상치 탐지율이 높아 부정 거래를 대부분 잘 탐지하였지만 정상 데이터까지 이상치로 분류하는 경우가 매우많아 정밀하지 않았음.